# Project 3: Regularized Regression for Minnesota Housing Prices

**Course:** CSCI 440/540 Artificial Intelligence and Neural Networks  
**Student:** Aubin Darcy Irakoze  

This notebook builds directly on Project 1 and extends the workflow by adding **regularized linear models**:
- Linear Regression
- Ridge Regression
- Lasso Regression
- ElasticNet Regression

It also satisfies the following requirements:
1. Exploratory Data Analysis (EDA)  
2. Data Preparation  
3. Model Training and Comparison  
4. Model Evaluation  
5. Final Pipeline and Prediction  
6. Discussion of overfitting, fairness, and interpretability

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Load the Dataset

This project uses the same Minnesota housing dataset from Project 1.

In [ ]:
DATA_PATH = Path("housing_data.csv")

housing = pd.read_csv(DATA_PATH)
housing.head()

In [ ]:
print("Shape:", housing.shape)
print("\nColumns:")
print(housing.columns.tolist())
print("\nMissing values:")
print(housing.isnull().sum())

## 3. Exploratory Data Analysis (EDA)

Requirements covereed are histograms, scatter plots, and a correlation matrix, as well as an **income-based stratification** variable.

In [ ]:
housing.describe()

In [ ]:
housing.hist(bins=30, figsize=(14, 10))
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_with_target = corr_matrix["median_house_value"].sort_values(ascending=False)
corr_with_target

In [ ]:
plt.figure(figsize=(9, 7))
plt.imshow(corr_matrix, cmap="coolwarm", interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
housing.plot(
    kind="scatter",
    x="median_income",
    y="median_house_value",
    alpha=0.25
)
plt.title("Median Income vs Median House Value")
plt.show()

### Income-based Stratification
I created income categories so the train/test split preserves similar income-group proportions.

In [ ]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

housing["income_cat"].value_counts().sort_index()

In [ ]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].copy()
    strat_test_set = housing.loc[test_index].copy()

for dataset in (strat_train_set, strat_test_set):
    dataset.drop(columns=["income_cat"], inplace=True)

print("Train shape:", strat_train_set.shape)
print("Test shape:", strat_test_set.shape)

## 4. Separate Features and Labels

In [ ]:
housing_train = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

housing_test = strat_test_set.drop("median_house_value", axis=1)
housing_test_labels = strat_test_set["median_house_value"].copy()

print("Training features shape:", housing_train.shape)
print("Training labels shape:", housing_labels.shape)

## 5. Data Preparation

Requirements covered here:
- Handle missing values with median imputation
- Feature scaling with `StandardScaler`
- Engineer new features:
  - `rooms_per_household`
  - `bedrooms_per_room`
  - `population_per_household`

This dataset contains only numeric columns, so there are no categorical features to one-hot encode.
We still use a reusable preprocessing pipeline so the workflow remains production-friendly.

In [ ]:
numeric_features = housing_train.columns.tolist()

def to_dataframe_with_numeric_columns(X):
    return pd.DataFrame(X, columns=numeric_features)

def add_extra_features(X):
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=numeric_features)
    X = X.copy()
    X["rooms_per_household"] = X["total_rooms"] / X["households"]
    X["bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]
    X["population_per_household"] = X["population"] / X["households"]
    return X

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("to_dataframe", FunctionTransformer(to_dataframe_with_numeric_columns, validate=False)),
    ("feature_engineering", FunctionTransformer(add_extra_features, validate=False)),
    ("scaler", StandardScaler())
])

full_preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_features)
])

prepared_sample = full_preprocessor.fit_transform(housing_train)
print("Prepared training matrix shape:", prepared_sample.shape)


## 6. Baseline and Regularized Models

We compare:
- Linear Regression
- Ridge
- Lasso
- ElasticNet

Evaluation metric: **RMSE**

In [ ]:
def rmse_cv(model_pipeline, X, y, cv=5):
    neg_mse_scores = cross_val_score(
        model_pipeline,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=cv
    )
    rmse_scores = np.sqrt(-neg_mse_scores)
    return rmse_scores

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso": Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=30000),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=30000),
}

cv_summary = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", model)
    ])

    scores = rmse_cv(pipeline, housing_train, housing_labels, cv=5)
    cv_summary.append({
        "Model": name,
        "Mean CV RMSE": scores.mean(),
        "Std CV RMSE": scores.std()
    })

cv_results_df = pd.DataFrame(cv_summary).sort_values("Mean CV RMSE")
cv_results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(cv_results_df["Model"], cv_results_df["Mean CV RMSE"])
plt.ylabel("Mean CV RMSE")
plt.title("Baseline Model Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning

We tune:
- `alpha` for Ridge
- `alpha` for Lasso
- `alpha` and `l1_ratio` for ElasticNet

In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", full_preprocessor),
    ("model", Ridge(random_state=RANDOM_STATE))
])

ridge_param_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_search = GridSearchCV(
    ridge_pipeline,
    ridge_param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)
ridge_search.fit(housing_train, housing_labels)

print("Best Ridge params:", ridge_search.best_params_)
print("Best Ridge CV RMSE:", np.sqrt(-ridge_search.best_score_))

In [ ]:
lasso_pipeline = Pipeline([
    ("preprocessor", full_preprocessor),
    ("model", Lasso(random_state=RANDOM_STATE, max_iter=30000))
])

lasso_param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
}

lasso_search = GridSearchCV(
    lasso_pipeline,
    lasso_param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)
lasso_search.fit(housing_train, housing_labels)

print("Best Lasso params:", lasso_search.best_params_)
print("Best Lasso CV RMSE:", np.sqrt(-lasso_search.best_score_))

In [ ]:
elastic_pipeline = Pipeline([
    ("preprocessor", full_preprocessor),
    ("model", ElasticNet(random_state=RANDOM_STATE, max_iter=30000))
])

elastic_param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
    "model__l1_ratio": [0.2, 0.5, 0.8]
}

elastic_search = GridSearchCV(
    elastic_pipeline,
    elastic_param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)
elastic_search.fit(housing_train, housing_labels)

print("Best ElasticNet params:", elastic_search.best_params_)
print("Best ElasticNet CV RMSE:", np.sqrt(-elastic_search.best_score_))

## 8. Final Model Evaluation on the Test Set

We evaluate:
- Linear Regression
- Best Ridge
- Best Lasso
- Best ElasticNet

We then compare their **test RMSE** values.

In [ ]:
final_models = {
    "LinearRegression": Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", LinearRegression())
    ]),
    "Ridge": ridge_search.best_estimator_,
    "Lasso": lasso_search.best_estimator_,
    "ElasticNet": elastic_search.best_estimator_
}

test_results = []

for name, model in final_models.items():
    model.fit(housing_train, housing_labels)
    predictions = model.predict(housing_test)
    rmse = np.sqrt(mean_squared_error(housing_test_labels, predictions))
    test_results.append({
        "Model": name,
        "Test RMSE": rmse
    })

test_results_df = pd.DataFrame(test_results).sort_values("Test RMSE")
test_results_df

In [ ]:
best_model_name = test_results_df.iloc[0]["Model"]
best_model = final_models[best_model_name]

best_model.fit(housing_train, housing_labels)
final_predictions = best_model.predict(housing_test)

print("Best model:", best_model_name)
print("Final Test RMSE:", np.sqrt(mean_squared_error(housing_test_labels, final_predictions)))

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(housing_test_labels, final_predictions, alpha=0.25)
plt.xlabel("Actual Median House Value")
plt.ylabel("Predicted Median House Value")
plt.title(f"Predicted vs Actual Values ({best_model_name})")

min_val = min(housing_test_labels.min(), final_predictions.min())
max_val = max(housing_test_labels.max(), final_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.tight_layout()
plt.show()

## 9. Coefficient Shrinkage Across Models

This section helps illustrate how regularization affects the learned coefficients:
- **Linear Regression:** no penalty
- **Ridge:** shrinks coefficients but usually keeps them non-zero
- **Lasso:** can drive some coefficients to zero
- **ElasticNet:** balances Ridge and Lasso behavior

In [ ]:
feature_names = numeric_features + [
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household"
]

coefficient_models = {
    "LinearRegression": Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", LinearRegression())
    ]),
    "Ridge": ridge_search.best_estimator_,
    "Lasso": lasso_search.best_estimator_,
    "ElasticNet": elastic_search.best_estimator_
}

coef_df = pd.DataFrame({"Feature": feature_names})

for name, model in coefficient_models.items():
    model.fit(housing_train, housing_labels)
    coefs = model.named_steps["model"].coef_
    coef_df[name] = coefs

coef_df

In [ ]:
coef_df_plot = coef_df.set_index("Feature")

plt.figure(figsize=(11, 6))
for col in coef_df_plot.columns:
    plt.plot(coef_df_plot.index, coef_df_plot[col], marker="o", label=col)

plt.axhline(0, linestyle="--")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Coefficient Value")
plt.title("Coefficient Shrinkage Across Linear and Regularized Models")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
coef_magnitude_df = coef_df.copy()
for col in ["LinearRegression", "Ridge", "Lasso", "ElasticNet"]:
    coef_magnitude_df[col] = coef_magnitude_df[col].abs()

coef_magnitude_df.sort_values(best_model_name, ascending=False).head(10)

## 10. Save the Final Pipeline with Joblib

Saving the final pipeline and using it for prediction on unseen data.

In [ ]:
OUTPUT_MODEL_PATH = Path("project3_best_model_pipeline.joblib")
joblib.dump(best_model, OUTPUT_MODEL_PATH)

print("Saved model to:", OUTPUT_MODEL_PATH.resolve())

## 11. Predict on New / Unseen Data

We take a few rows from the test set, remove the target, and generate predictions using the saved pipeline.

In [ ]:
loaded_model = joblib.load(OUTPUT_MODEL_PATH)

new_data = housing_test.iloc[:5].copy()
new_predictions = loaded_model.predict(new_data)

prediction_df = new_data.copy()
prediction_df["Predicted_Median_House_Value"] = new_predictions
prediction_df["Actual_Median_House_Value"] = housing_test_labels.iloc[:5].values

prediction_df

## 12. Interpretation and Reflection

### How regularization affects generalization
Regularization helps control model complexity by penalizing large coefficients. This reduces the chance that the model fits noise in the training data, which improves performance on unseen data.

### How regularization affects interpretability
- **Ridge** improves stability when features are correlated, but usually keeps all predictors.
- **Lasso** is easier to interpret because it can push some coefficients to zero, effectively performing feature selection.
- **ElasticNet** is useful when we want both shrinkage and sparse selection.

### Risks of overfitting in real estate prediction
Real estate data can contain correlated or noisy variables. Without regularization, a model may overfit training patterns that do not hold in new neighborhoods or future market conditions. This can produce unstable price estimates.

### Fairness considerations
Predictive real estate models may unintentionally reflect or reinforce socioeconomic inequality. For example, variables such as income can be highly predictive but may also encode structural disparities. Regularization can improve stability, but it does not remove fairness concerns by itself. Responsible deployment still requires careful feature review, transparency, and awareness of downstream impact.

### Final takeaway
Project 3 extends Project 1 by showing that:
- regularization improves robustness,
- coefficient shrinkage helps interpretability,
- and a reusable pipeline makes the workflow more reliable and production-ready.